# Option 2 — Elevated Pressure Profile on GPS Route

Draws a **filled pressure profile** along the route — like a cross-sectional
area chart that follows the path on the map.

- **Right foot** fills upward (to the right of travel) in blue  
- **Left foot** fills downward (to the left of travel) in orange  
- Height of the filled area ∝ pressure at that moment  
- A `Polygon` is drawn for each foot: forward along the elevated edge,  
  then backward along the GPS baseline, closing a filled shape

```
         ████ right foot filled area ████
         █                              █
GPS ─────·──────────────────────────────·─────
         █                              █
         ████  left foot filled area ████
```

**Output:** `pressure_profile_map.html`

## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
from scipy.ndimage import gaussian_filter1d
import folium
import warnings
warnings.filterwarnings('ignore')

print(f'folium {folium.__version__}')

folium 0.20.0


## Cell 2 — Configuration

In [2]:
CSV_PATH = 'Kristian_dry_0425.csv'
GPX_PATH = 'activity_22645980458.gpx'

PRESSURE_COLS    = [f'pressure_{i:02d}' for i in range(1, 13)]
GPS_TOLERANCE_MS = 2000

# ── Profile amplitude ─────────────────────────────────────────────────────────
# Max perpendicular height of the filled area at pressure = 1.0
# 0.00018 deg ~ 20 m at lat=42. Increase for more dramatic profiles.
MAX_OFFSET_DEG = 0.00022   # slightly larger than waveform for visual clarity

# ── Smoothing ─────────────────────────────────────────────────────────────────
# Higher sigma = smoother profile silhouette
SMOOTH_SIGMA = 4

# ── Chunk size for polygons ───────────────────────────────────────────────────
# Split the route into segments of this many GPX points per polygon.
# Smaller = more polygons, smoother colour gradient possible.
# Larger = fewer DOM elements, faster rendering.
CHUNK_SIZE = 20

# ── Colours ───────────────────────────────────────────────────────────────────
RIGHT_COLOR   = '#1F77B4'
RIGHT_FILL    = '#1F77B4'
LEFT_COLOR    = '#FF7F0E'
LEFT_FILL     = '#FF7F0E'
BASE_COLOR    = '#888888'
FILL_OPACITY  = 0.45

print('Config ready.')
print(f'  Max profile height: {MAX_OFFSET_DEG * 111000:.0f} m at full pressure')
print(f'  Chunk size: {CHUNK_SIZE} GPX points per polygon')

Config ready.
  Max profile height: 24 m at full pressure
  Chunk size: 20 GPX points per polygon


## Cell 3 — Load Data

In [3]:
raw = pd.read_csv(CSV_PATH, low_memory=False)
raw = raw[raw['corrupt'] == 0].copy()
raw['timestamp']      = pd.to_numeric(raw['timestamp'], errors='coerce')
raw['total_pressure'] = raw[PRESSURE_COLS].sum(axis=1)

def parse_gpx(path):
    tree = ET.parse(path)
    root = tree.getroot()
    ns   = {'gpx': 'http://www.topografix.com/GPX/1/1'}
    rows = []
    for trkpt in root.findall('.//gpx:trkpt', ns):
        dt = pd.to_datetime(trkpt.find('gpx:time', ns).text.strip(), utc=True)
        rows.append({'lat': float(trkpt.get('lat')),
                     'lon': float(trkpt.get('lon')),
                     'unix_ms': int(dt.value // 1_000_000)})
    return pd.DataFrame(rows)

gpx_df = parse_gpx(GPX_PATH)
print(f'Insole: {len(raw):,} rows | GPX: {len(gpx_df)} points')

Insole: 51,267 rows | GPX: 440 points


## Cell 4 — Build Per-Foot Pressure Signal & Normalise

In [4]:
def pressure_signal(sole_id):
    return (
        raw[raw['sole_id'] == sole_id]
        .groupby('timestamp', as_index=False)['total_pressure'].mean()
        .sort_values('timestamp').reset_index(drop=True)
    )

def attach_to_gpx(gpx, pressure_df, col_name):
    merged = pd.merge_asof(
        gpx.sort_values('unix_ms'),
        pressure_df.rename(columns={'timestamp': 'unix_ms',
                                    'total_pressure': col_name}),
        on='unix_ms', direction='nearest', tolerance=GPS_TOLERANCE_MS,
    )
    col = merged[col_name]
    col = col.fillna(col.rolling(7, center=True, min_periods=1).median())
    col = col.ffill().bfill().fillna(col.mean() if col.notna().any() else 0.0)
    merged[col_name] = col
    return merged

gpx_r = attach_to_gpx(gpx_df.copy(), pressure_signal(1), 'pressure_right')
gpx_l = attach_to_gpx(gpx_df.copy(), pressure_signal(2), 'pressure_left')
gpx_both = gpx_r.merge(gpx_l[['unix_ms','pressure_left']], on='unix_ms', how='left')

# Clean any NaNs introduced by the left-merge
for col in ['pressure_right', 'pressure_left']:
    gpx_both[col] = gpx_both[col].ffill().bfill().fillna(0.0)

# Normalise 0-1 across both feet so heights are comparable
p_max = max(gpx_both['pressure_right'].max(), gpx_both['pressure_left'].max())
p_min = min(gpx_both['pressure_right'].min(), gpx_both['pressure_left'].min())
gpx_both['pnorm_right'] = (gpx_both['pressure_right'] - p_min) / (p_max - p_min + 1e-10)
gpx_both['pnorm_left']  = (gpx_both['pressure_left']  - p_min) / (p_max - p_min + 1e-10)

# Guard before smoothing
gpx_both['pnorm_right'] = gpx_both['pnorm_right'].ffill().bfill().fillna(0.0)
gpx_both['pnorm_left']  = gpx_both['pnorm_left'].ffill().bfill().fillna(0.0)
gpx_both['psmooth_right'] = gaussian_filter1d(gpx_both['pnorm_right'].values, sigma=SMOOTH_SIGMA)
gpx_both['psmooth_left']  = gaussian_filter1d(gpx_both['pnorm_left'].values,  sigma=SMOOTH_SIGMA)

nan_counts = gpx_both[['psmooth_right','psmooth_left']].isna().sum()
print(f'GPX + pressure: {len(gpx_both)} rows')
print(f'  psmooth_right range: {gpx_both["psmooth_right"].min():.3f} – {gpx_both["psmooth_right"].max():.3f}')
print(f'  psmooth_left  range: {gpx_both["psmooth_left"].min():.3f} – {gpx_both["psmooth_left"].max():.3f}')
print(f'  NaNs remaining — right: {nan_counts["psmooth_right"]}  left: {nan_counts["psmooth_left"]}  (must be 0)')

GPX + pressure: 440 rows
  psmooth_right range: 0.349 – 0.887
  psmooth_left  range: 0.300 – 0.784
  NaNs remaining — right: 0  left: 0  (must be 0)


## Cell 5 — Geometry Functions

### Elevated edge
Each GPS point is offset perpendicular to the route by `pressure × MAX_OFFSET_DEG`.

### Filled polygon
For a chunk of N GPS points, the filled polygon is:
```
  elevated[0] → elevated[1] → ... → elevated[N-1]   (top edge, forward)
  baseline[N-1] → ... → baseline[0]                 (bottom edge, backward)
  close at elevated[0]
```
This traces the area between the route and the pressure profile.

In [5]:
def compute_perp_unit(lats, lons, i):
    """Central-difference perpendicular unit vector at index i."""
    n = len(lats)
    lat_scale = np.cos(np.radians(lats[i]))
    if 0 < i < n - 1:
        dlat = lats[i+1] - lats[i-1]
        dlon = (lons[i+1] - lons[i-1]) * lat_scale
    elif i == 0:
        dlat = lats[1] - lats[0]
        dlon = (lons[1] - lons[0]) * lat_scale
    else:
        dlat = lats[-1] - lats[-2]
        dlon = (lons[-1] - lons[-2]) * lat_scale
    mag = np.sqrt(dlat**2 + dlon**2) + 1e-12
    return -dlon/mag, dlat/mag/lat_scale


def elevated_edge(gpx, pressure_col, side, max_offset=MAX_OFFSET_DEG):
    """
    Compute the elevated profile edge for one foot.
    side=+1 right of travel, side=-1 left.
    Returns list of (lat, lon) — guaranteed NaN-free.
    """
    lats = gpx['lat'].values
    lons = gpx['lon'].values
    pres = np.nan_to_num(gpx[pressure_col].values, nan=0.0)  # safety guard
    coords = []
    for i in range(len(lats)):
        plat, plon = compute_perp_unit(lats, lons, i)
        amp = float(pres[i]) * max_offset * side
        coords.append((float(lats[i] + plat * amp),
                        float(lons[i] + plon * amp)))
    return coords


def make_polygon_chunks(baseline_coords, elevated_coords, chunk_size=CHUNK_SIZE):
    """
    Split the route into overlapping chunks and return polygon vertex lists.
    Each polygon = forward along elevated + backward along baseline.
    Overlap of 1 point between chunks ensures no gaps.
    """
    n = len(baseline_coords)
    polygons = []
    for start in range(0, n - 1, chunk_size - 1):
        end = min(start + chunk_size, n)
        elev_chunk = elevated_coords[start:end]
        base_chunk = baseline_coords[start:end]
        # Forward along elevated edge, backward along baseline
        poly = elev_chunk + base_chunk[::-1]
        polygons.append(poly)
    return polygons


# Compute elevated edges
baseline_coords = list(zip(gpx_both['lat'], gpx_both['lon']))
elev_right = elevated_edge(gpx_both, 'psmooth_right', side=+1)
elev_left  = elevated_edge(gpx_both, 'psmooth_left',  side=-1)

# Build polygon chunks
polys_right = make_polygon_chunks(baseline_coords, elev_right)
polys_left  = make_polygon_chunks(baseline_coords, elev_left)

print(f'Right foot: {len(polys_right)} polygons')
print(f'Left  foot: {len(polys_left)} polygons')
print(f'Each polygon has up to {CHUNK_SIZE * 2} vertices')

Right foot: 24 polygons
Left  foot: 24 polygons
Each polygon has up to 40 vertices


## Cell 6 — Build Folium Map

Four layers:
1. **Grey baseline** — actual GPS route  
2. **Blue filled area** — right foot profile (right of travel direction)  
3. **Orange filled area** — left foot profile (left of travel direction)  
4. **Elevated edge lines** — sharp outline of each profile for clarity

In [6]:
centre_lat = gpx_df['lat'].mean()
centre_lon = gpx_df['lon'].mean()

m = folium.Map(
    location=[centre_lat, centre_lon],
    zoom_start=18,
    tiles='CartoDB positron',
)

# ── Baseline route ────────────────────────────────────────────────────────────
folium.PolyLine(
    baseline_coords,
    color=BASE_COLOR, weight=2, opacity=0.7,
    tooltip='GPS baseline route',
).add_to(m)

# ── Right foot filled profile ─────────────────────────────────────────────────
right_fg = folium.FeatureGroup(name='Right foot profile', show=True)

for poly in polys_right:
    folium.Polygon(
        locations=poly,
        color=RIGHT_COLOR,
        fill=True,
        fill_color=RIGHT_FILL,
        fill_opacity=FILL_OPACITY,
        weight=0.5,
        opacity=0.3,
    ).add_to(right_fg)

# Sharp elevated edge line on top of the fill
folium.PolyLine(
    elev_right,
    color=RIGHT_COLOR, weight=2, opacity=0.9,
    tooltip='Right foot pressure profile',
).add_to(right_fg)

right_fg.add_to(m)

# ── Left foot filled profile ──────────────────────────────────────────────────
left_fg = folium.FeatureGroup(name='Left foot profile', show=True)

for poly in polys_left:
    folium.Polygon(
        locations=poly,
        color=LEFT_COLOR,
        fill=True,
        fill_color=LEFT_FILL,
        fill_opacity=FILL_OPACITY,
        weight=0.5,
        opacity=0.3,
    ).add_to(left_fg)

folium.PolyLine(
    elev_left,
    color=LEFT_COLOR, weight=2, opacity=0.9,
    tooltip='Left foot pressure profile',
).add_to(left_fg)

left_fg.add_to(m)

# ── Start / end markers ───────────────────────────────────────────────────────
folium.Marker(
    [gpx_df['lat'].iloc[0],  gpx_df['lon'].iloc[0]],
    icon=folium.Icon(color='green', icon='play', prefix='fa'),
    tooltip='Walk start',
).add_to(m)
folium.Marker(
    [gpx_df['lat'].iloc[-1], gpx_df['lon'].iloc[-1]],
    icon=folium.Icon(color='red', icon='stop', prefix='fa'),
    tooltip='Walk end',
).add_to(m)

# ── Legend ────────────────────────────────────────────────────────────────────
legend_html = f"""
<div style="
    position:fixed; bottom:30px; left:30px; z-index:1000;
    background:white; padding:12px 16px; border-radius:8px;
    box-shadow:0 2px 8px rgba(0,0,0,0.25); font-family:Arial; font-size:12px;
">
  <b style='font-size:13px;'>Pressure Profile</b><br>
  <span style='color:{BASE_COLOR};'>&#9644;</span> GPS baseline route<br>
  <span style='color:{RIGHT_COLOR};'>&#9646;</span> Right foot (fills right)<br>
  <span style='color:{LEFT_COLOR};'>&#9646;</span> Left foot (fills left)<br>
  <hr style='margin:6px 0;'>
  Profile height ∝ foot pressure<br>
  Max height: {MAX_OFFSET_DEG*111000:.0f} m at full pressure<br>
  Smoothing sigma: {SMOOTH_SIGMA}<br>
  <i style='font-size:11px;'>Toggle layers top-right</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)

m.save('pressure_profile_map.html')
print('Saved → pressure_profile_map.html')
m

Saved → pressure_profile_map.html
